# Frontier RAG Benchmark (HotpotQA)

Grounded in HotpotQA (Yang et al., EMNLP 2018) — distractor setting.

**Cutting-edge angle:** not another single RAG flavor — a *system* that routes compute:

| Method | Role |
|--------|------|
| **FrontierRAG** | Classify → BM25+dense RRF → cross-encoder rerank → CRAG grade → escalate to hybrid |
| Adaptive / BM25+dense / Rerank | Ablations of Frontier building blocks |
| Semantic / Hybrid / GraphRAG modes | Classic baselines |
| HippoRAG 2 / LightRAG | Opt-in knowledge-graph RAG families |

**Reading scenario scores:** `0.357 \| 0.298` = same method on bridge vs comparison — not two mystery metrics.

See `results/routing_cheatsheet.md` and `choose_over_examples.csv` for data-driven choose-A-over-B.

In [ ]:
# ── Config ─────────────────────────────────────────────────
BENCHMARK = "hotpotqa"   # "hotpotqa" (paper) | "wikipedia" (custom)
N_HOTPOT = 12
FETCH_OR_BUILD = True
RUN_BENCHMARK = False    # results already in results/; set True to re-run
REUSE_INDEXES = True
METHODS = [
    "frontier_rag",          # Adaptive + CRAG escalate (cutting-edge system)
    "adaptive_rag",
    "hybrid_dense_sparse",
    "rerank_semantic",
    "semantic_rag",
    "hybrid_rag",
    "lazygraph_rag",         # GraphRAG fast/basic — not MS LazyGraphRAG
    # "hippo_rag", "light_rag",  # opt-in; slow OpenIE on 3B
]
# ───────────────────────────────────────────────────────────

import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
load_dotenv(PROJECT_ROOT / ".env")

from rag_benchmark import (
    BenchmarkConfig, BenchmarkRunner, create_tracked_client,
    fetch_corpus, build_hotpot_subset,
)
from rag_benchmark.charts import METHOD_LABELS, render_notebook_dashboard
from rag_benchmark.hybrid_rag import HybridRAG

# Guardrail: hybrid must not double-generate
src = (PROJECT_ROOT / "src/rag_benchmark/hybrid_rag.py").read_text()
assert "self.semantic.retrieve(" in src and "self.semantic.query(" not in src
print("Hybrid guardrail OK: retrieve-only + one fusion call")

config = BenchmarkConfig.from_yaml(PROJECT_ROOT)
config.project_root = PROJECT_ROOT
config.reuse_indexes = REUSE_INDEXES
print(f"Backend={config.llm_backend} chat={config.chat_model} embed={config.embedding_model}")

## 1. Build eval corpus

**HotpotQA distractor** builds a *closed* paragraph set from the paper JSON (no full-Wikipedia crawl/index).

In [ ]:
if BENCHMARK == "hotpotqa":
    if FETCH_OR_BUILD:
        paths = build_hotpot_subset(
            project_root=PROJECT_ROOT, n_questions=N_HOTPOT, prefer_hard=True
        )
        print(paths["meta"])
    config.corpus_dir = PROJECT_ROOT / "data" / "corpus_hotpot"
    config.qa_path = PROJECT_ROOT / "data" / "qa" / "hotpot_eval.json"
    # Separate Chroma collection + GraphRAG workspace so we don't mix wiki indexes
    config.semantic_collection = "hotpot_semantic"
    config.graph_workspace = PROJECT_ROOT / "graphrag_workspaces" / "hotpot"
    config.lazy_workspace = PROJECT_ROOT / "graphrag_workspaces" / "hotpot_lazy"
    config.max_documents = 10_000  # use all Hotpot paragraphs written for the subset
else:
    config.corpus_dir = PROJECT_ROOT / "data" / "corpus"
    config.qa_path = PROJECT_ROOT / "data" / "qa" / "eval_questions.json"
    if FETCH_OR_BUILD:
        titles = [
            "Albert Einstein", "Python (programming language)", "Machine learning",
            "Artificial intelligence", "Solar System", "Mars", "Jupiter",
            "World War II", "Neural network", "United States",
        ]
        fetch_corpus(titles, config.corpus_dir, max_chars=10_000)

n_docs = len(list(config.corpus_dir.glob("*.txt")))
print(f"Corpus: {config.corpus_dir} ({n_docs} docs)")
print(f"QA:     {config.qa_path}")

## 2. Run methods

GraphRAG uses **fast** indexing locally (spaCy NLP graph) — `standard` LLM extraction often fails on 3B models during community reports.

In [ ]:
results_dir = config.results_dir()

if RUN_BENCHMARK:
    runner = BenchmarkRunner(config, create_tracked_client(config))
    print(f"{len(runner.questions)} questions")
    for q in runner.questions[:5]:
        print(f"  [{q.query_type}] {q.question[:80]}")
    results = runner.run_all(methods=METHODS)
    saved = runner.save_results(results)
else:
    saved = {
        "summary_df": pd.read_csv(results_dir / "summary.csv"),
        "accuracy_df": pd.read_csv(results_dir / "accuracy_results.csv"),
        "latency_df": pd.read_csv(results_dir / "latency_results.csv"),
        "token_df": pd.read_csv(results_dir / "token_results.csv"),
        "scenario_df": pd.read_csv(results_dir / "scenario_results.csv")
        if (results_dir / "scenario_results.csv").exists() else pd.DataFrame(),
    }
    results = None
    print("Loaded saved results")

## Key finding (HotpotQA distractor, n=12)

| Scenario | Winner (latest run) | Why |
|----------|---------------------|-----|
| **local** (comparison / single-fact) | **Vector + rerank** (~0.56) | Phrase match; stays under p95≤5s |
| **hybrid** (bridge / multi-hop) | **FrontierRAG** (~0.41) | Grade + escalate to hybrid when retrieval is weak |
| Interactive default | **Vector + rerank** (~0.43 overall) | Best quality among methods with p95 ≤ 5s |

### Decoding “fast/basic … 0.357 / 0.298”

Same method, **two Hotpot scenarios** (mean composite quality):

| Number | Scenario | Meaning |
|--------|----------|---------|
| **0.357** | `hybrid` column | Hotpot **bridge** (multi-hop) questions |
| **0.298** | `local` column | Hotpot **comparison** questions |

GraphRAG fast/basic is decent on multi-hop for a cheap NLP index, but loses to vector RAG on simple comparisons. It is **not** Microsoft LazyGraphRAG.

## Demo: question ID → actual question

Hotpot IDs are opaque hex strings. This catalog maps **Q1–Q12** to the full question, gold answer, type, and how metrics behave on average across methods.

Yellow highlight in the table below (via `em_rate == 0`): models often score on the LLM judge / contains, but almost never **exact-match** the gold span.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
results_dir = ROOT / "results"

catalog = pd.read_csv(results_dir / "question_catalog.csv")
# Demo-friendly columns
demo = catalog[
    [
        "label",
        "question_id",
        "hotpot_type",
        "question",
        "gold_answer",
        "em_rate",
        "avg_llm_judge",
        "avg_token_f1",
        "contains_rate",
        "best_method",
        "best_composite",
    ]
].copy()
demo["question_id"] = demo["question_id"].str.slice(0, 10) + "…"

def _style_em_fail(row):
    # Highlight questions where EM never fires but judge still sees signal
    if row.em_rate == 0 and row.avg_llm_judge >= 0.2:
        return ["background-color: #fff3cd"] * len(row)
    return [""] * len(row)

display(Markdown("### Question catalog"))
display(demo.style.apply(_style_em_fail, axis=1).format({
    "em_rate": "{:.0%}",
    "avg_llm_judge": "{:.2f}",
    "avg_token_f1": "{:.2f}",
    "contains_rate": "{:.0%}",
    "best_composite": "{:.2f}",
}))

display(Markdown("Full IDs: `results/question_catalog.csv`"))
for _, r in catalog.iterrows():
    print(f"{r.label}  {r.question_id}  [{r.hotpot_type}]")
    print(f"    Q: {r.question}")
    print(f"    Gold: {r.gold_answer}\n")


## Metric autopsy: why LLM judge ≠ EM / F1

Composite = mean(**LLM judge**, **token F1**, **exact match**, **contains**).

| Metric | Strictness | What fails on generative GraphRAG |
|--------|------------|-----------------------------------|
| **Exact match** | Strict string identity (Hotpot-normalized) | Prose / missing honorifics (`DSC`) → EM=0 |
| **Token F1** | Token overlap | Long answers dilute precision |
| **Contains** | Fuzzy gold substring | Often passes when EM fails |
| **LLM judge** | 0–1 factual overlap | Rewards paraphrase; **same 3B as generator** — soft |

**Demo takeaway:** GraphRAG fast/basic can look “good” on judge/contains and “dead” on EM/F1. Prefer contains+judge for UX; EM+F1 for extractive Hotpot-style scoring.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

acc = pd.read_csv(results_dir / "accuracy_results.csv")
breakdown = pd.read_csv(results_dir / "metric_breakdown_by_method.csv")

display(Markdown("### Mean metrics by method"))
display(breakdown.set_index("method").round(3).sort_values("llm_judge_score", ascending=False))

# Disagreement stats
n = len(acc)
hi_judge_no_em = int(((acc.llm_judge_score >= 0.5) & (acc.exact_match == False)).sum())
contains_no_em = int(((acc.contains_answer == True) & (acc.exact_match == False)).sum())
display(Markdown(
    f"- **{hi_judge_no_em}/{n}** rows: LLM judge ≥ 0.5 but EM = 0\n"
    f"- **{contains_no_em}/{n}** rows: contains gold but EM = 0\n"
    f"- When EM=0: mean judge={acc.loc[~acc.exact_match, 'llm_judge_score'].mean():.2f}, "
    f"mean F1={acc.loc[~acc.exact_match, 'token_f1'].mean():.2f}"
))

# Chart: judge vs EM by method
plot_df = breakdown.set_index("method")[
    ["llm_judge_score", "contains_answer", "token_f1", "exact_match"]
].sort_values("llm_judge_score", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
plot_df.plot(kind="barh", ax=ax)
ax.set_xlabel("Mean score (0–1)")
ax.set_title("Hotpot n=12 — LLM judge vs lexical metrics by method")
ax.set_xlim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# Per-question judge − EM gap
cat = pd.read_csv(results_dir / "question_catalog.csv")
cat["judge_minus_em"] = cat["avg_llm_judge"] - cat["em_rate"]
fig2, ax2 = plt.subplots(figsize=(9, 4.5))
ax2.barh(
    cat["label"] + " (" + cat["hotpot_type"] + ")",
    cat["judge_minus_em"],
    color="#c9a227",
)
ax2.set_xlabel("Avg LLM judge − EM rate")
ax2.set_title("Where lexical metrics under-credit the model")
plt.tight_layout()
plt.show()


## Recent RAG benchmarks (2024–2026) — beyond Hotpot

HotpotQA (2018) is still the default multi-hop smoke test, but the last two years added suites that stress **trust**, **multi-doc news**, **synthetic AutoE**, and **when graphs help**.

| Benchmark | Year | Stresses | Vs this harness |
|-----------|------|----------|-----------------|
| [MultiHop-RAG](https://arxiv.org/abs/2401.15391) (COLM’24) | 2024 | 2,556 multi-hop news queries; evidence across 2–4 docs | Better “real RAG” multi-doc retrieval than Wikipedia spans |
| [CRAG](https://arxiv.org/abs/2406.04744) (Meta / KDD Cup’24) | 2024 | 4.4k QA; web+KG APIs; dynamism / popularity / complexity | Hallucination & freshness — not just EM |
| [BenchmarkQED](https://github.com/microsoft/benchmark-qed) (Microsoft) | 2025 | AutoQ / AutoE / AutoD; pairwise LLM-as-judge | Scalable eval; stronger than same-model judge |
| [GraphRAG-Bench](https://github.com/GraphRAG-Bench/GraphRAG-Benchmark) (ICLR’26) | 2025–26 | Domain reasoning; graph build → retrieve → generate | Answers *when GraphRAG beats vector* |
| RGB / RECALL | 2023–24 | Noise & counterfactuals | Generation robustness; weak on multi-hop retrieval |

**Recommended next steps for this repo:** keep Hotpot for routing bake-offs → add MultiHop-RAG for retrieval → CRAG-style trust scores → BenchmarkQED AutoE with a **stronger separate judge**.

Full write-up: `results/recent_rag_benchmarks.md`

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path

bench_md = results_dir / "recent_rag_benchmarks.md"
display(Markdown(bench_md.read_text(encoding="utf-8")))


## When to choose which RAG (from *your* data)

Cutting-edge production pattern is **Adaptive-RAG routing** — not crowning one stack. Below is generated from `results/accuracy_results.csv` (Hotpot n=12).



In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
from rag_benchmark.decision_playbook import build_decision_artifacts
from rag_benchmark.examples_playbook import format_examples_markdown

qa_path = Path(config.qa_path) if "config" in dir() else ROOT / "data" / "qa" / "hotpot_eval.json"
results_dir = Path(RESULTS_DIR) if "RESULTS_DIR" in dir() else ROOT / "results"

paths = build_decision_artifacts(results_dir, qa_path)
display(Markdown(Path(paths["cheatsheet"]).read_text(encoding="utf-8")))
display(Markdown("### Choose-A-over-B table"))
display(__import__("pandas").read_csv(paths["examples"]))
display(Markdown("### Pairwise win rates (BenchmarkQED-lite)"))
display(__import__("pandas").read_csv(paths["pairwise"]).head(12))
display(Markdown("---\n### Method intuition cards"))
display(Markdown(format_examples_markdown()))


## Engineering scorecard

What teams actually need: **routing** (which method for which query), **p95 latency**, **tokens/query**, and a default production pick — not just a single overall average.

In [ ]:
from IPython.display import Image, Markdown, display
from rag_benchmark.engineering import (
    build_engineering_scorecard,
    print_engineering_briefing,
    save_engineering_scorecard,
)

if "summary_df" not in dir() or summary_df is None:
    summary_df = pd.read_csv(RESULTS_DIR / "summary.csv")
    accuracy_df = pd.read_csv(RESULTS_DIR / "accuracy_results.csv")
    scenario_df = (
        pd.read_csv(RESULTS_DIR / "scenario_results.csv")
        if (RESULTS_DIR / "scenario_results.csv").exists()
        else pd.DataFrame()
    )

scorecard = build_engineering_scorecard(
    summary_df=summary_df,
    scenario_df=scenario_df,
    accuracy_df=accuracy_df,
)
eng_paths = save_engineering_scorecard(scorecard, RESULTS_DIR)
print_engineering_briefing(scorecard)

display(Markdown("### Method scorecard"))
display(pd.DataFrame(scorecard["methods"])[
    [
        c
        for c in [
            "label",
            "mean_composite_score",
            "usable_answer_rate",
            "tokens_per_query",
            "quality_per_1k_tokens",
            "p95_query_latency_seconds",
            "meets_latency_slo",
        ]
        if c in pd.DataFrame(scorecard["methods"]).columns
    ]
])
display(Markdown("### Routing recommendations"))
display(pd.DataFrame(scorecard["routing_by_query_type"]))
if Path(eng_paths["chart"]).exists():
    display(Image(filename=str(eng_paths["chart"])))
display(Markdown(Path(eng_paths["briefing"]).read_text(encoding="utf-8")))


## 3. Charts + leaderboard (quality, tokens, scenarios)

In [ ]:
frames = render_notebook_dashboard(saved=saved, results_dir=results_dir)
summary = frames["summary"]
cols = [c for c in [
    "label", "mean_composite_score", "mean_llm_judge", "mean_token_f1",
    "exact_match_rate", "total_tokens", "tokens_per_query",
    "mean_query_latency_seconds",
] if c in summary.columns]
display(summary.sort_values("mean_composite_score", ascending=False)[cols].round(3))

In [ ]:
scenario = frames.get("scenario")
if scenario is not None and not scenario.empty:
    pivot = scenario.pivot_table(
        index="label", columns="query_type", values="mean_composite_score", observed=False
    ).round(3)
    print("Quality by Hotpot-style scenario (bridge→hybrid, comparison→local):")
    display(pivot)

print("\nToken efficiency (quality per 1k tokens):")
eff = summary.copy()
eff["quality_per_1k_tokens"] = (
    eff["mean_composite_score"] / (eff["total_tokens"].clip(lower=1) / 1000)
).round(3)
display(eff[["label", "mean_composite_score", "total_tokens", "quality_per_1k_tokens"]]
        .sort_values("quality_per_1k_tokens", ascending=False))

## 4. Sample answers

In [ ]:
if results:
    for result in results:
        print(f"\n=== {METHOD_LABELS.get(result.method, result.method)} | tokens={result.ledger.total().total_tokens} ===")
        for row in result.answers[:2]:
            print(f"[{row.get('query_type')}] Q: {row['question']}")
            print(f"A: {row['answer'][:350]}\n")
else:
    print("Set RUN_BENCHMARK=True for live answers.")